In [ ]:
!pip install -q streamlit langchain-groq langchain-community langchain-core langchain-huggingface langchain-text-splitters faiss-cpu pypdf python-dotenv sentence-transformers

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
%%writefile app.py
import streamlit as st
import os
import json
import re
import tempfile
import warnings
from dotenv import load_dotenv
from groq import Groq

warnings.filterwarnings("ignore")

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

st.set_page_config(page_title="Tutor PDF + Anki Chat", layout="wide", page_icon="🧠")

# ==========================================
# GERENCIAMENTO DE ESTADO (SESSION STATE)
# ==========================================
if "api_key" not in st.session_state:
    st.session_state.api_key = os.getenv("GROQ_API_KEY", "")
if "messages" not in st.session_state:
    st.session_state.messages = [
        {
            "role": "assistant",
            "content": "Olá! Insira sua **Groq API Key** na barra lateral e faça o upload de um PDF para começarmos. Quando quiser fixar o conteúdo, digite **'gerar perguntas'**, **'flashcards'** ou **'anki'** no chat!"
        }
    ]
if "retriever" not in st.session_state:
    st.session_state.retriever = None
if "full_text" not in st.session_state:
    st.session_state.full_text = ""
if "active_files" not in st.session_state:
    st.session_state.active_files = []
if "anki_mode" not in st.session_state:
    st.session_state.anki_mode = False
if "flashcards" not in st.session_state:
    st.session_state.flashcards = []
if "current_card" not in st.session_state:
    st.session_state.current_card = 0
if "show_answer" not in st.session_state:
    st.session_state.show_answer = False

# ==========================================
# FUNÇÕES AUXILIARES E CACHE
# ==========================================
@st.cache_resource
def get_embeddings():
    return HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

@st.cache_data(ttl=300)
def get_available_models(api_key):
    """Busca dinamicamente os modelos ativos na conta Groq do usuário."""
    fallback = ["openai/gpt-oss-120b", "openai/gpt-oss-20b", "llama-3.1-8b-instant"]
    if not api_key:
        return fallback
    try:
        client = Groq(api_key=api_key)
        models = client.models.list().data
        # Filtra apenas modelos de texto/chat (ignora whisper/tts/guard)
        ignorar = ("whisper", "tts", "guard", "safeguard", "playai")
        validos = [m.id for m in models if not any(x in m.id.lower() for x in ignorar)]
        validos.sort()
        # Coloca openai/gpt-oss-120b no topo se estiver disponível
        if "openai/gpt-oss-120b" in validos:
            validos.remove("openai/gpt-oss-120b")
            validos.insert(0, "openai/gpt-oss-120b")
        return validos if validos else fallback
    except Exception:
        return fallback

def load_llm(api_key, id_model, temperature):
    return ChatGroq(
        groq_api_key=api_key,
        model=id_model,
        temperature=temperature,
        max_retries=2,
    )

def clean_thinking(res):
    if "</think>" in res:
        res = res.split("</think>")[-1].strip()
    return res

def process_pdfs(uploaded_files):
    docs = []
    for uploaded_file in uploaded_files:
        with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp_file:
            tmp_file.write(uploaded_file.getvalue())
            tmp_path = tmp_file.name

        loader = PyPDFLoader(tmp_path)
        docs.extend(loader.load())
        os.remove(tmp_path)

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = text_splitter.split_documents(docs)

    full_text = "\n\n".join([c.page_content for c in chunks[:25]])

    embeddings = get_embeddings()
    vectorstore = FAISS.from_documents(chunks, embeddings)
    return vectorstore.as_retriever(search_kwargs={"k": 5}), full_text

def generate_flashcards(llm, context_text, num_cards=5):
    prompt = f"""
    Você é um especialista em criar flashcards de estudo (estilo Anki) para fixação de conteúdo.
    Com base EXCLUSIVAMENTE no texto abaixo, gere {num_cards} perguntas curtas e diretas com respostas igualmente curtas e objetivas.

    Retorne APENAS um array JSON válido, sem texto adicional e sem formatação markdown, seguindo exatamente esta estrutura:
    [
        {{"pergunta": "Pergunta curta 1?", "resposta": "Resposta curta e direta 1."}},
        {{"pergunta": "Pergunta curta 2?", "resposta": "Resposta curta e direta 2."}}
    ]

    Texto base:
    {context_text[:12000]}
    """
    res = llm.invoke(prompt)
    content = clean_thinking(res.content)

    try:
        match = re.search(r'\[.*\]', content, re.DOTALL)
        if match:
            return json.loads(match.group(0))
        return json.loads(content)
    except Exception as e:
        st.error(f"Erro ao estruturar os flashcards: {e}")
        return []

# ==========================================
# BARRA LATERAL (CONFIGURAÇÕES E ARQUIVOS)
# ==========================================
with st.sidebar:
    st.header("🔑 Configurações")
    sidebar_key = st.text_input(
        "Groq API Key",
        value=st.session_state.api_key,
        type="password",
        placeholder="Cole sua chave gsk_..."
    )
    if sidebar_key != st.session_state.api_key:
        st.session_state.api_key = sidebar_key

    if st.session_state.api_key:
        st.success("✅ API Key conectada!")
    else:
        st.warning("⚠️ Insira sua Groq API Key")

    st.divider()
    st.header("🤖 Modelo & Anki")
    modelos_ativos = get_available_models(st.session_state.api_key)
    id_model = st.selectbox("Escolha o Modelo (Ativos na conta)", modelos_ativos)
    temperature = st.slider("Temperatura", 0.1, 1.5, 0.4, 0.1)
    num_cards = st.slider("Qtd. de Perguntas no Anki", 3, 15, 5)

    st.divider()
    st.header("📂 Arquivos PDF")
    uploaded_files = st.file_uploader(
        "Envie seu(s) arquivo(s) PDF",
        type=["pdf"],
        accept_multiple_files=True
    )

    if uploaded_files:
        current_names = [f.name for f in uploaded_files]
        if current_names != st.session_state.active_files:
            with st.spinner("Processando novo PDF..."):
                retriever, full_text = process_pdfs(uploaded_files)
                st.session_state.retriever = retriever
                st.session_state.full_text = full_text
                old_files = st.session_state.active_files
                st.session_state.active_files = current_names

                lista_nomes = ", ".join([f"**{n}**" for n in current_names])
                if not old_files:
                    msg_sistema = f"📄 **Arquivo carregado:** {lista_nomes}. Agora sou especialista neste material! O que deseja perguntar?"
                else:
                    msg_sistema = f"🔄 **Mudança de arquivo detectada!** O contexto do chat foi alterado para: {lista_nomes}."

                st.session_state.messages.append({"role": "assistant", "content": msg_sistema})
                st.toast(f"Contexto atualizado: {', '.join(current_names)}", icon="📄")

    st.divider()
    col_b1, col_b2 = st.columns(2)
    with col_b1:
        if st.button("🧹 Limpar", use_container_width=True):
            st.session_state.messages = [
                {"role": "assistant", "content": "Histórico limpo! Como posso ajudar?"}
            ]
            st.rerun()
    with col_b2:
        if st.session_state.retriever and not st.session_state.anki_mode:
            if st.button("🗂️ Anki", type="primary", use_container_width=True):
                if not st.session_state.api_key:
                    st.error("Insira a API Key!")
                else:
                    with st.spinner("Gerando flashcards..."):
                        llm = load_llm(st.session_state.api_key, id_model, temperature)
                        cards = generate_flashcards(llm, st.session_state.full_text, num_cards)
                        if cards:
                            st.session_state.flashcards = cards
                            st.session_state.current_card = 0
                            st.session_state.show_answer = False
                            st.session_state.anki_mode = True
                            st.rerun()

# ==========================================
# ÁREA PRINCIPAL
# ==========================================
st.title("Tutor Especialista & Modo Anki 🧠")

if st.session_state.active_files:
    st.info(f"📚 **Documento(s) ativo(s) no momento:** {', '.join(st.session_state.active_files)}")
else:
    st.warning("📂 **Nenhum PDF carregado.** Utilize o painel à esquerda para enviar o seu arquivo PDF.")

# MODO ANKI VS CHAT
if st.session_state.anki_mode:
    col_title, col_exit = st.columns([3, 1])
    with col_title:
        st.subheader("🗂️ Modo Anki: Fixação de Conteúdo")
    with col_exit:
        if st.button("❌ Sair do Anki", type="secondary", use_container_width=True):
            st.session_state.anki_mode = False
            st.session_state.show_answer = False
            st.session_state.messages.append(
                {"role": "assistant", "content": "✅ **Você saiu do Modo Anki.** Voltamos ao chat normal!"}
            )
            st.rerun()

    cards = st.session_state.flashcards
    idx = st.session_state.current_card

    if cards:
        st.progress((idx + 1) / len(cards), text=f"Cartão {idx + 1} de {len(cards)}")

        with st.container(border=True):
            st.markdown(f"### ❓ Pergunta:\n#### {cards[idx]['pergunta']}")

            if st.session_state.show_answer:
                st.divider()
                st.markdown(f"### 💡 Resposta:\n#### :green[{cards[idx]['resposta']}]")

        col1, col2, col3 = st.columns(3)

        with col1:
            if st.button("⬅️ Anterior", disabled=(idx == 0), use_container_width=True):
                st.session_state.current_card -= 1
                st.session_state.show_answer = False
                st.rerun()

        with col2:
            btn_label = "🙈 Ocultar Resposta" if st.session_state.show_answer else "👁️ Ver Resposta"
            if st.button(btn_label, type="primary", use_container_width=True):
                st.session_state.show_answer = not st.session_state.show_answer
                st.rerun()

        with col3:
            if st.button("Próximo ➡️", disabled=(idx == len(cards) - 1), use_container_width=True):
                st.session_state.current_card += 1
                st.session_state.show_answer = False
                st.rerun()

        if st.session_state.show_answer:
            st.write("")
            c_errei, c_bom, c_facil = st.columns(3)
            with c_errei:
                if st.button("🔴 Errei (Repetir)", use_container_width=True):
                    st.session_state.flashcards.append(cards[idx])
                    st.session_state.current_card += 1
                    st.session_state.show_answer = False
                    st.rerun()
            with c_bom:
                if st.button("🟡 Bom", use_container_width=True) and idx < len(cards) - 1:
                    st.session_state.current_card += 1
                    st.session_state.show_answer = False
                    st.rerun()
            with c_facil:
                if st.button("🟢 Fácil", use_container_width=True) and idx < len(cards) - 1:
                    st.session_state.current_card += 1
                    st.session_state.show_answer = False
                    st.rerun()

else:
    for msg in st.session_state.messages:
        with st.chat_message(msg["role"]):
            st.markdown(msg["content"])

    user_input = st.chat_input("Tire dúvidas sobre o PDF ou digite 'gerar perguntas' para abrir o Anki...")

    if user_input:
        if not st.session_state.api_key:
            st.error("🔑 Insira sua Groq API Key na barra lateral.")
            st.stop()
        if not st.session_state.retriever:
            st.warning("📂 Envie um arquivo PDF na barra lateral primeiro!")
            st.stop()

        st.session_state.messages.append({"role": "user", "content": user_input})
        with st.chat_message("user"):
            st.markdown(user_input)

        llm = load_llm(st.session_state.api_key, id_model, temperature)

        gatilhos_anki = [
            "anki", "flashcard", "flashcards", "gerar perguntas",
            "gere perguntas", "perguntas e respostas", "fixar", "quiz"
        ]
        pediu_anki = any(gatilho in user_input.lower() for gatilho in gatilhos_anki)

        if pediu_anki:
            with st.chat_message("assistant"):
                with st.spinner("Criando perguntas e abrindo o Anki..."):
                    retrieved_docs = st.session_state.retriever.invoke(user_input)
                    context_anki = "\n\n".join([doc.page_content for doc in retrieved_docs])
                    if len(context_anki) < 500:
                        context_anki = st.session_state.full_text

                    cards = generate_flashcards(llm, context_anki, num_cards)

                    if cards:
                        st.session_state.flashcards = cards
                        st.session_state.current_card = 0
                        st.session_state.show_answer = False
                        st.session_state.anki_mode = True
                        st.session_state.messages.append(
                            {"role": "assistant", "content": f"🗂️ Preparei **{len(cards)} flashcards**! Entrando no Modo Anki..."}
                        )
                        st.rerun()
                    else:
                        st.error("Não consegui gerar os cartões. Tente novamente.")
        else:
            with st.chat_message("assistant"):
                with st.spinner("Consultando o PDF..."):
                    retrieved_docs = st.session_state.retriever.invoke(user_input)
                    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

                    history_text = "\n".join(
                        [f"{m['role']}: {m['content']}" for m in st.session_state.messages[-6:-1]]
                    )

                    prompt_rag = ChatPromptTemplate.from_template("""
                    Você é um tutor especialista nos documentos enviados pelo usuário.
                    Responda de forma clara, didática e objetiva com base exclusivamente no contexto abaixo.
                    Se a resposta não estiver no documento, avise educadamente.

                    Histórico recente:
                    {history}

                    Contexto do PDF:
                    {context}

                    Pergunta: {question}
                    """)

                    chain = prompt_rag | llm
                    res = chain.invoke({
                        "history": history_text,
                        "context": context,
                        "question": user_input
                    })

                    clean_res = clean_thinking(res.content)
                    st.markdown(clean_res)
                    st.session_state.messages.append({"role": "assistant", "content": clean_res})

Writing app.py


In [ ]:
!pkill -f streamlit
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb >/dev/null 2>&1

!streamlit run app.py --server.port 8501 --server.enableCORS false --server.enableXsrfProtection false &>/content/logs.txt &
!cloudflared tunnel --url http://localhost:8501

2026-11-16T19:37:14Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-16T19:37:14Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-16T19:37:19Z INF +--------------------------------------------------------------------------------------------+
2026-11-16T19:37:19Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-16T19:37:19Z INF |  https://superb-divide-hired-projection.trycloudflare.